# RLHF Fine-Tuning with LoRA + DPO

Parameter-efficient fine-tuning: attach LoRA adapters, train with
Direct Preference Optimization (`DPOTrainer`), and run S5 safety checks on
the output. `backward()` requires the C backend.

In [ ]:
import SneppX_ALG as S
from SneppX_ALG import (
    Transformer, AdamW, Tokenizer, DPOTrainer,
    LoRAConfig, LoRALinear, S5RLHFSafety, S5OutputVerifier,
)
HAS_C = S._HAS_C_BACKEND
print('C backend:', HAS_C)

## 1. Base model + LoRA adapters

In [ ]:
base = Transformer(vocab_size=300, dim=128, num_heads=4, num_layers=4,
                   ffn_dim=256, max_seq_len=64)
lora_cfg = LoRAConfig(r=8, alpha=32, dropout=0.1)
base.lm_head = LoRALinear(base.lm_head, r=lora_cfg.r, alpha=lora_cfg.alpha)
trainable = sum(p.numel for n, p in base.named_parameters() if 'lora_' in n)
print('trainable params:', trainable)

## 2. DPO training

In [ ]:
trainer = DPOTrainer(
    policy=base,
    ref_policy=None,
    beta=0.1,
    optimizer=AdamW(filter(lambda p: p.requires_grad, base.parameters()), lr=5e-4),
)

prefs = [
    ([1, 2, 3], [10, 11, 12], [99, 98]),
    ([4, 5, 6], [20, 21], [88, 87, 86]),
]
for prompt, chosen, rejected in prefs:
    if not HAS_C:
        print('C backend required for DPO backward - skipping')
        break
    loss = trainer.dpo_loss(
        prompt_input_ids=prompt,
        chosen_input_ids=chosen,
        rejected_input_ids=rejected,
    )
    trainer.optimizer.zero_grad()
    loss.backward(); trainer.optimizer.step()
    print('dpo loss:', float(loss.data))

## 3. GRPO alternative

In [ ]:
from SneppX_ALG import GRPOTrainer
grpo = GRPOTrainer(
    policy=base,
    optimizer=AdamW(base.parameters(), lr=3e-4),
    num_generations=4, beta=0.01,
)
print('GRPO ready:', grpo is not None)

## 4. Safety-check generated output (S5)

In [ ]:
verifier = S5OutputVerifier()
safe = S5RLHFSafety(allowed_topics=['science', 'technology'])
# After generation:
#   verifier.check(text) -> bool
#   safe.is_safe(text)    -> bool
sample = 'SNEPPX is a secure neural engine.'
print('safe?', safe.is_safe(sample) if HAS_C else 'requires C backend')